# Step 2: Club-context features (destination manager + team strength)

Adds two kinds of club-level features to the transfers built in `01_join_and_baselines.ipynb` and checks
whether they beat the baselines **on the same held-out split** (`data/processed/split.csv`).

| Feature | Meaning | Time window (t = transfer date) |
|---|---|---|
| `from_elo`, `to_elo` | ClubElo rating of the selling / buying club | latest snapshot on or before t − 1 day |
| `elo_step` | `to_elo − from_elo` (positive = step up) | same |
| `dest_mgr_tenure_days` | days the buying club's manager had been in charge | games strictly before t |
| `dest_new_mgr_180d` | 1 if that manager took over < 180 days before t | games strictly before t |
| `dest_mgr_tenure_censored` | 1 if the spell began before the data does (tenure is a minimum) | games strictly before t |

**Diagnostics only, never model features** (they use games after the transfer, which a scout can't know):
`dest_mgr_differs_after` (first game after t has a different manager), `dest_mgr_change_in_window`
(the manager changed during the 365-day label window).

Code lives in `src/features/club_context.py`; assumptions about name matching and manager spells are documented there.
Needs `data/raw/clubelo/EloRatings.csv` (run `python src/data/download_data.py`) and the outputs of notebook 01.

In [ ]:
# Download the raw data into data/raw/ (skips files you already have; ~250 MB on a fresh clone).
# Uses the pinned sources in src/data/download_data.py; raises DownloadError listing anything that failed.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data" / "download_data.py").is_file())
sys.path.insert(0, str(ROOT))
from src.data.download_data import download_all
download_all(quiet=True)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No data/raw folder found in {Path.cwd()} or its parents")
sys.path.insert(0, str(ROOT))
from src.features import club_context as cc

RAW, OUT, INTERIM = ROOT / "data/raw", ROOT / "data/processed", ROOT / "data/interim"
SEED = 42
LABEL_DAYS = 365
ELO_ACTIVE = ("2017-07-01", "2020-07-01")  # ClubElo clubs considered when matching names
pd.set_option("display.width", 160)

data = pd.read_csv(OUT / "transfers_joined.csv", parse_dates=["transfer_date"])
split = pd.read_csv(OUT / "split.csv", parse_dates=["transfer_date"])
clubs = pd.read_csv(RAW / "transfermarkt/clubs.csv.gz")
transfers = pd.read_csv(RAW / "transfermarkt/transfers.csv.gz")
games = pd.read_csv(RAW / "transfermarkt/games.csv.gz", parse_dates=["date"])
elo = cc.load_elo(RAW / "clubelo/EloRatings.csv")
print(f"{len(data)} transfers, {elo.club.nunique()} ClubElo clubs, {elo.date.nunique()} Elo snapshots")

276 transfers, 984 ClubElo clubs, 629 Elo snapshots


## 1. Link Transfermarkt clubs to ClubElo names
Fuzzy match within the same country, accepted only with a clear winner, plus hand-checked overrides
(`ELO_OVERRIDES`). Clubs with no Elo (reserve teams, leagues ClubElo doesn't cover) stay unlinked rather
than guessed. **Eyeball the lowest-scoring fuzzy matches below.**

In [2]:
club_ids = pd.unique(pd.concat([data.from_club_id, data.to_club_id]).dropna())
link = cc.match_clubs_to_elo(club_ids, clubs, transfers, elo, ELO_ACTIVE)
INTERIM.mkdir(parents=True, exist_ok=True)
link.to_csv(INTERIM / "club_elo_link.csv", index=False)
print(link.method.value_counts().to_string())
print("\nLowest-scoring fuzzy matches:")
display(link[link.method == "fuzzy"].nsmallest(10, "score"))
print("Not linked:")
display(link[~link.method.isin(["fuzzy", "override"])])

method
fuzzy         114
override       15
no_country      9

Lowest-scoring fuzzy matches:


,club_id,tm_name,country,elo_club,score,runner_up_score,method
78,1162,SM Caen,FRA,Caen,86.363636,53.846154,fuzzy
106,3057,Standard Liège,BEL,Standard,86.363636,48.000000,fuzzy
6,618,AS Saint-Étienne,FRA,St Etienne,86.956522,56.000000,fuzzy
17,44,Hertha BSC,GER,Hertha,87.500000,40.909091,fuzzy
61,1420,Angers SCO,FRA,Angers,87.500000,55.555556,fuzzy
77,897,Deportivo de La Coruña,ESP,La Coruna,87.500000,51.724138,fuzzy
53,681,Real Sociedad,ESP,Sociedad,88.095238,58.333333,fuzzy
111,367,Rayo Vallecano,ESP,Vallecano,89.130435,54.545455,fuzzy
115,366,Real Valladolid CF,ESP,Valladolid,90.000000,69.230769,fuzzy
124,683,Olympiakos Syndesmos Filathlon Peiraios,GRE,Olympiakos,90.000000,53.333333,fuzzy


Not linked:


,club_id,tm_name,country,elo_club,score,runner_up_score,method
30,11972,Villarreal CF B,NaN,NaN,NaN,NaN,no_country
31,8880,Alavés B,NaN,NaN,NaN,NaN,no_country
32,6767,RM Castilla,NaN,NaN,NaN,NaN,no_country
33,10331,FC Porto B,NaN,NaN,NaN,NaN,no_country
39,8733,Celta Fortuna,NaN,NaN,NaN,NaN,no_country
66,36630,HB CFFC,NaN,NaN,NaN,NaN,no_country
69,33304,DL Yifang,NaN,NaN,NaN,NaN,no_country
94,18075,CD Maldonado,NaN,NaN,NaN,NaN,no_country
97,515,Without Club,NaN,NaN,NaN,NaN,no_country


## 2. Compute the features

In [3]:
cg = cc.club_manager_games(games)
feat = pd.concat([cc.elo_features(data, elo, link), cc.manager_features(data, cg, LABEL_DAYS)], axis=1)
df = pd.concat([data, feat], axis=1)

NEW = ["from_elo", "to_elo", "elo_step", "dest_mgr_tenure_days", "dest_new_mgr_180d", "dest_mgr_tenure_censored"]
DIAG = ["dest_mgr_differs_after", "dest_mgr_change_in_window"]
print("Coverage (share of transfers with a value):")
print(df[NEW + DIAG].notna().mean().round(3).to_string())
df[NEW + DIAG].describe().T.round(2)

Coverage (share of transfers with a value):
from_elo                     0.967
to_elo                       1.000
elo_step                     0.967
dest_mgr_tenure_days         1.000
dest_new_mgr_180d            1.000
dest_mgr_tenure_censored     1.000
dest_mgr_differs_after       1.000
dest_mgr_change_in_window    1.000


,count,mean,std,min,25%,50%,75%,max
from_elo,267.0,1686.55,132.51,1409.96,1595.41,1672.40,1761.27,2027.39
to_elo,276.0,1684.41,131.87,1342.35,1597.34,1679.42,1741.28,2027.35
elo_step,267.0,-2.65,153.84,-429.04,-99.14,11.26,98.41,397.78
dest_mgr_tenure_days,276.0,556.73,554.73,1.00,201.00,330.00,865.25,2353.00
dest_new_mgr_180d,276.0,0.22,0.42,0.00,0.00,0.00,0.00,1.00
dest_mgr_tenure_censored,276.0,0.12,0.33,0.00,0.00,0.00,0.00,1.00
dest_mgr_differs_after,276.0,0.21,0.41,0.00,0.00,0.00,0.00,1.00
dest_mgr_change_in_window,276.0,0.56,0.50,0.00,0.00,1.00,1.00,1.00


**Known weakness of the manager features:** `dest_mgr_differs_after` is the share of transfers where
the first game after the move has a different manager than the last game before it. Those are mostly
summer appointments that were already public at transfer time, so for them the tenure feature describes
the *previous* manager.

In [4]:
print(f"Manager at first game after transfer differs from last game before: {df.dest_mgr_differs_after.mean():.1%}")
print(f"Manager changed within the 365-day label window (diagnostic):      {df.dest_mgr_change_in_window.mean():.1%}")
print(f"Tenure is a lower bound (spell started before data):              {df.dest_mgr_tenure_censored.mean():.1%}")

Manager at first game after transfer differs from last game before: 21.0%
Manager changed within the 365-day label window (diagnostic):      55.8%
Tenure is a lower bound (spell started before data):              12.0%


## 3. Do the features relate to playing time?
Simple checks before any model. Correlations are on all 278 transfers, so treat them as descriptive only.

In [5]:
print("Correlation with y:")
print(df[NEW + DIAG].corrwith(df.y).round(3).to_string())

df["elo_step_bin"] = pd.qcut(df.elo_step, 4, labels=["big step down", "step down", "step up", "big step up"])
print("\nMean y by size of the Elo step:")
print(df.groupby("elo_step_bin", observed=True).y.agg(["mean", "count"]).round(3).to_string())
print("\nMean y by manager change in the label window (diagnostic, post-transfer):")
print(df.groupby("dest_mgr_change_in_window").y.agg(["mean", "count"]).round(3).to_string())

Correlation with y:
from_elo                     0.012
to_elo                      -0.103
elo_step                    -0.093
dest_mgr_tenure_days        -0.043
dest_new_mgr_180d            0.022
dest_mgr_tenure_censored     0.080
dest_mgr_differs_after       0.021
dest_mgr_change_in_window   -0.136

Mean y by size of the Elo step:
                mean  count
elo_step_bin               
big step down  0.500     67
step down      0.492     67
step up        0.536     66
big step up    0.442     67

Mean y by manager change in the label window (diagnostic, post-transfer):
                            mean  count
dest_mgr_change_in_window              
0.0                        0.534    122
1.0                        0.454    154


## 4. Baselines vs. club context, same split and CV as notebook 01
The constant-mean and base linear-regression **test** MSEs must reproduce notebook 01's (`baseline_results.csv`), which confirms
the comparison uses the identical split. CV numbers can differ in the 4th decimal because the training rows are in a
different order here, so the 5-fold splits differ.

In [6]:
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

key = ["player_id", "wy_id", "transfer_date", "to_club_id"]
df = df.merge(split[key + ["split"]], on=key, how="left", validate="one_to_one")
assert df.split.notna().all(), "every transfer must be in split.csv"
train_df, test_df = df[df.split == "train"], df[df.split == "test"]

P90 = [c for c in data.columns if c.endswith("_p90")]
BASE_NUM = ["age", "log_mv", "prev_share_tm", "wy_share", "pass_acc"] + P90
CAT = ["position"]

def linreg(num):
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return make_pipeline(pre, LinearRegression())

models = {
    "constant mean (sanity)": (DummyRegressor(strategy="mean"), BASE_NUM),
    "linear regression: base": (linreg(BASE_NUM), BASE_NUM),
    "linear regression: club context only": (linreg(NEW), NEW),
    "linear regression: base + club context": (linreg(BASE_NUM + NEW), BASE_NUM + NEW),
}
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=SEED)
rows = []
for name, (mdl, num) in models.items():
    X_tr, X_te = train_df[num + CAT], test_df[num + CAT]
    cv_mse = -cross_val_score(mdl, X_tr, train_df.y, cv=cv, scoring="neg_mean_squared_error")
    mdl.fit(X_tr, train_df.y)
    test_mse = mean_squared_error(test_df.y, mdl.predict(X_te))
    rows.append((name, cv_mse.mean(), cv_mse.std(), test_mse, np.sqrt(test_mse)))
results = pd.DataFrame(rows, columns=["model", "cv_mse_mean", "cv_mse_std", "test_mse", "test_rmse"])
print(f"train n = {len(train_df)}, test n = {len(test_df)}")
display(results.round(4))

lr = models["linear regression: club context only"][0]
coef = pd.Series(lr[-1].coef_, index=lr[0].get_feature_names_out()).sort_values(key=abs, ascending=False)
print("\nClub-context-only coefficients (standardised features):")
print(coef.round(4).to_string())

train n = 220, test n = 56


,model,cv_mse_mean,cv_mse_std,test_mse,test_rmse
0,constant mean (sanity),0.0819,0.0114,0.0973,0.3119
1,linear regression: base,0.0874,0.0139,0.1012,0.3181
2,linear regression: club context only,0.0893,0.0156,0.1007,0.3174
3,linear regression: base + club context,0.0841,0.0189,0.0940,0.3066



Club-context-only coefficients (standardised features):
num__to_elo                      -0.1381
num__elo_step                     0.1358
num__from_elo                     0.1240
cat__position_Goalkeeper          0.0608
num__dest_mgr_tenure_days        -0.0451
cat__position_Attack             -0.0440
num__dest_mgr_tenure_censored     0.0380
cat__position_Defender           -0.0223
num__dest_new_mgr_180d           -0.0174
cat__position_Midfield            0.0054
num__missingindicator_from_elo    0.0003
num__missingindicator_elo_step    0.0003


In [7]:
df.drop(columns=["elo_step_bin", "split"]).to_csv(OUT / "transfers_club_context.csv", index=False)
results.to_csv(OUT / "club_context_results.csv", index=False)
print(f"Saved transfers_club_context.csv and club_context_results.csv to {OUT}; club_elo_link.csv to {INTERIM}")

Saved transfers_club_context.csv and club_context_results.csv to /home/slage/Coding/Deep Learning/Project/Deep-Learning-Scouting/data/processed; club_elo_link.csv to /home/slage/Coding/Deep Learning/Project/Deep-Learning-Scouting/data/interim
